# WattWise AI - Step 2: Anomaly Detection with Isolation Forest
This notebook details the training, threshold tuning, synthetic anomaly injection, and baseline benchmark comparison of the WattWise AI Anomaly Detector.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import build_features
from src.anomaly import train_isolation_forest, score, inject_anomalies, zscore_baseline, evaluate_anomaly_detection

## 1. Load Processed Hourly Data & Compute Engineered Features

In [ ]:
df_hourly = pd.read_parquet('../data/processed/hourly.parquet')
df_features = build_features(df_hourly.head(5000))
df_features.head()

## 2. Quantitative Evaluation: Isolation Forest vs Z-Score Baseline

In [ ]:
metrics_df, details = evaluate_anomaly_detection(df_hourly, contamination=0.03, train_split_ratio=0.75, reports_dir='../reports')
print(metrics_df.to_string(index=False))

## 3. Visualizing Detected Anomalies on Sample Timeline

In [ ]:
model = joblib.load('../models/isolation_forest.joblib')
scored_df = score(model, df_features)
h_id = scored_df['household_id'].iloc[0]
sample = scored_df[scored_df['household_id'] == h_id].iloc[:168]

plt.figure(figsize=(13, 4.5))
plt.plot(sample['timestamp'], sample['kwh'], label='Hourly Consumption', color='#1f77b4', lw=1.5)
anomalies = sample[sample['anomaly_flag']]
plt.scatter(anomalies['timestamp'], anomalies['kwh'], color='red', s=50, label='Flagged Anomaly', zorder=5)
plt.title(f'Isolation Forest Detected Anomalies - Household {h_id}', fontsize=12, fontweight='bold')
plt.xlabel('Timestamp')
plt.ylabel('kWh')
plt.legend()
plt.tight_layout()
plt.show()